# Phase 3 Visual Audit: Multi-Tier Sentiment & Cultural Themes

This thin visual audit notebook tests and compares the 3 sentiment tiers:
1. **Tier 1 (Lexicon):** VADER compound score in `[-1.0, +1.0]`.
2. **Tier 2 (Transformer):** RoBERTa head+tail chunked contextual sentiment.
3. **Tier 3 (Structured LLM):** Gemini 2.5 Flash culture relevance, 15-theme taxonomy, actor role, and key quote extraction.
4. **Model Comparison:** Analyzes cases of model agreement and disagreement on complex workplace rhetoric.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on Python sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.common.config import EngineConfig
from src.common.azure_storage import StorageManager
from src.enrichment.vader_analyzer import VaderAnalyzer
from src.enrichment.roberta_analyzer import RobertaAnalyzer
from src.enrichment.gemini_enricher import GeminiEnricher
from src.enrichment.pipeline import EnrichmentPipeline
from src.ingest.standardizer import build_canonical_record, to_canonical_dataframe

config = EngineConfig.load()
vader = VaderAnalyzer()
roberta = RobertaAnalyzer()
gemini = GeminiEnricher(config, mock_mode=True)
print("✅ All 3 sentiment analyzers successfully loaded.")

### 1. Comparative Benchmark on Diverse Workplace Statements

In [ ]:
test_cases = [
    ("I am so thrilled leadership decided to mandate 5 days in office with zero notice. What an absolute joy!", ["Amazon"]),
    ("Morale in our building completely collapsed after the layoff wave. Everyone is updating their LinkedIn.", ["Google"]),
    ("Our manager is genuinely great. Gives us full autonomy, zero micromanagement, and protects us from upper management politics.", ["Meta"]),
    ("Just another day surviving PIP quotas and 70-hour weeks. Burnout is through the roof.", ["Amazon"]),
    ("The company offers competitive total compensation, but the culture is cutthroat and backstabbing.", ["Microsoft"]),
]

records = []
for i, (text, comps) in enumerate(test_cases):
    v_res = vader.analyze_text(text)
    r_res = roberta.analyze_text(text)
    g_res = gemini.enrich_record(f"test_{i}", text, comps, subreddit="cscareerquestions", date_str="2024-09")
    
    records.append({
        "Statement": text[:60] + "...",
        "Company": comps[0],
        "VADER Score": v_res["vader_score"],
        "VADER Label": v_res["vader_label"],
        "RoBERTa Score": r_res["roberta_score"],
        "RoBERTa Label": r_res["roberta_label"],
        "LLM Score": g_res.sentiment_score,
        "LLM Themes": g_res.themes,
        "LLM Role": g_res.actor_role,
    })

df_comp = pd.DataFrame(records)
display(df_comp)

### 2. Disagreement Analysis (Sarcasm & Complex Sentiment)

In [ ]:
sarcasm_row = df_comp.iloc[0]
print("Sarcasm Test Case:")
print(f"Text: {test_cases[0][0]}")
print(f"VADER:   {sarcasm_row['VADER Score']} ({sarcasm_row['VADER Label']}) -> Misled by words 'thrilled', 'joy'")
print(f"RoBERTa: {sarcasm_row['RoBERTa Score']} ({sarcasm_row['RoBERTa Label']}) -> Contextual sensitivity")
print(f"LLM:     {sarcasm_row['LLM Score']} -> Themes: {sarcasm_row['LLM Themes']}")

### 3. Checkpoint & Resume Validation

In [ ]:
checkpoint_ids = gemini.load_checkpoint_ids()
print(f"Total cached checkpoint records in JSONL: {len(checkpoint_ids)}")
assert len(checkpoint_ids) >= 5, "Checkpoints must be persisted to avoid duplicate LLM billing!"